# Experiment Notebook: DQN Upgrade

Goal: test a neural Q-network upgrade against tabular and baseline policies on the hard stress grid.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str((Path.cwd().parent / "src").resolve()))
from ml_engine import research_upgrade_lab as lab

lab.set_seed(42)
df = lab.load_clean_dataset(Path.cwd())
weights = {"carbon": 0.35, "jct": 0.20, "tail": 0.25, "preempt": 0.10, "starvation": 0.10}

q_tab = lab.train_q_domain_randomized(df=df, episodes=140, reward_weights=weights, hard_bias=0.70)

if lab.torch is None:
    raise RuntimeError("PyTorch is not available for DQN training in this environment")

net = lab.train_dqn(df=df, episodes=140, reward_weights=weights)

rows = []
for seed in range(20):
    for noise in [15.0, 20.0]:
        for cong in ["moderate", "high"]:
            for cap in [6, 8]:
                rows.append(lab.run_policy_inference(df, "srtf", seed, None, noise, cong, cap, reward_weights=weights))
                rows.append(lab.run_policy_inference(df, "carbon", seed, None, noise, cong, cap, reward_weights=weights))
                rows.append(lab.run_policy_inference(df, "fcfs", seed, None, noise, cong, cap, reward_weights=weights))

                q_row = lab.run_policy_inference(df, "q_tab", seed, q_tab, noise, cong, cap, reward_weights=weights)
                q_row["policy"] = "q_tabular"
                rows.append(q_row)

                dqn_row = lab.run_dqn_policy(df, net, seed, noise, cong, cap, reward_weights=weights)
                rows.append(dqn_row)

detail = pd.DataFrame(rows)
scored = lab.compute_overall_score_robust(detail, weights)
hard = lab.summarize_hard_slice(scored)
display(hard.round(4))